# Evaluating Attribute Annotations

In [2]:
import pandas as pd
import numpy as np

In [3]:
pd.set_option('display.max_rows', None)
pd.set_option("display.max_colwidth", None)

In [5]:
df_combined = pd.read_csv("../data/evaluation/annotation_with_complete_with_curation_and_llm.csv", encoding="utf-8-sig")

### Standardized Attributes

In [453]:
from sklearn.metrics import cohen_kappa_score
def eval_standardized(suff1, suff2, label, na=False):
    for attr in ["ReasonForCheck", "VehicleType", "SubstanceUsed", "LicenseStatus", "TypeOfRoad", "RefusedTesting", "Measuringmethods"]:
        if na:
            df_clean = df_combined.replace(np.nan,"n/a", inplace=False)
        else:
            df_clean = df_combined.replace("n/a", np.nan, inplace=False)
            df_clean = df_clean.dropna(subset=[f"{attr}.{suff1}", f"{attr}.{suff2}"])
        df_clean = df_clean.applymap(
        lambda x: (
            "; ".join(sorted(set(i.strip() for i in str(x).split(";") if i.strip())))
            if pd.notnull(x) else x
        )
    )
        print(f"Evaluating all {len(df_clean)} records for {attr}")
        print("\nAccuracy")
        accuracy = (df_clean[f"{attr}.{suff1}"] == df_clean[f"{attr}.{suff2}"]).sum()/len(df_clean)
        #print(f'Annot1 vs. Annot2: {(df2[f"{attr}.value.0"] == df2[f"{attr}.value.1"]).sum()/len(df2)}')
        print(f'{label} {accuracy}')
        
        print("\nCohen's Kappa")
        #kappa = cohen_kappa_score(df2[f"{attr}.value.0"], df2[f"{attr}.value.1"])
        #print(f'Annot1 vs. Annot2: {kappa}')
        kappa = cohen_kappa_score(df_clean[f"{attr}.{suff2}"], df_clean[f"{attr}.{suff1}"])
        print(f'{label} {kappa}\n\n')
        print(f'{attr} & {accuracy} & {kappa}')


In [492]:
from sklearn.metrics import cohen_kappa_score
def eval_all_standardized():
    for attr in ["ReasonForCheck", "VehicleType", "SubstanceUsed", "LicenseStatus", "TypeOfRoad", "RefusedTesting", "Measuringmethods"]:

        #df_clean = df_combined.replace(np.nan,"n/a", inplace=False)
        #df_clean = df_combined.replace("n/a", np.nan, inplace=False)
        #df_clean = df_clean.dropna(subset=[f"{attr}.value.curated", f"{attr}.llm"])
        df_clean = df_combined.replace(np.nan,"n/a", inplace=False).replace("None","n/a", inplace=False)
        df_clean = df_clean.applymap(
        lambda x: (
            "; ".join(sorted(set(i.strip() for i in str(x).split(";") if i.strip())))
            if pd.notnull(x) else x
        )
    )
        accuracy_llm = (df_clean[f"{attr}.value.curated"] == df_clean[f"{attr}.llm"]).sum()/len(df_clean)*100
        kappa_llm = cohen_kappa_score(df_clean[f"{attr}.llm"], df_clean[f"{attr}.value.curated"])*100
        accuracy_annot = (df_clean[f"{attr}.value.0"] == df_clean[f"{attr}.value.1"]).sum()/len(df_clean)*100
        kappa_annot = cohen_kappa_score(df_clean[f"{attr}.value.1"], df_clean[f"{attr}.value.0"])*100
        print(f'{attr}  & {accuracy_annot:.2f} & {kappa_annot:.2f} & {accuracy_llm:.2f} & {kappa_llm:.2f}')
eval_all_standardized()

ReasonForCheck  & 53.96 & 25.11 & 36.63 & 22.66
VehicleType  & 74.75 & 33.07 & 91.09 & 63.27
SubstanceUsed  & 66.34 & 34.54 & 72.28 & 36.06
LicenseStatus  & 93.07 & 58.44 & 92.08 & 61.45
TypeOfRoad  & 53.96 & 35.96 & 47.03 & 32.05
RefusedTesting  & 98.02 & 84.73 & 98.02 & 85.64
Measuringmethods  & 66.83 & 35.79 & 74.26 & 39.35


### Span Attributes

In [445]:
import re
from sklearn.metrics import cohen_kappa_score
def eval_span(suf1, suf2, label, na=False):
    for attr in ["TimeOfOffense", "DateOfOffense", "MeasuringTime", "AmountUsed"]:
        if na:
            df_clean = df_combined.replace(np.nan,"n/a", inplace=False)
        else:
            df_clean = df_combined.replace("n/a", np.nan, inplace=False)
            df_clean = df_clean.dropna(subset=[f"{attr}.{suf1}", f"{attr}.{suf2}"])
        df_clean = df_clean.applymap(
        lambda x: (
            "; ".join(sorted(set(i.strip() for i in str(x).split(";") if i.strip())))
            if pd.notnull(x) else x
        )
    )
        # Eval match between digit characters only
        df_clean = df_clean.applymap(
        lambda x: re.sub(r"\D", "", str(x)) if pd.notnull(x) else x
    )
        print(f"Evaluating all {len(df_clean)} records for {attr}")
        print("\nAccuracy")
        #print(f'Annot1 vs. Annot2: {(df2[f"{attr}.value.0"] == df2[f"{attr}.value.1"]).sum()/len(df2)}')
        print(f'{label} {(df_clean[f"{attr}.{suf1}"].apply(lambda x: str(x).split(";")[0].strip()) == df_clean[f"{attr}.{suf2}"]).sum()/len(df_clean)}\n\n')
    


In [486]:
from sklearn.metrics import cohen_kappa_score
def eval_all_span():
    for attr in ["TimeOfOffense", "DateOfOffense", "MeasuringTime", "AmountUsed"]:

        df_clean = df_combined.replace(np.nan,"n/a", inplace=False)
        #df_clean = df_combined.replace("n/a", np.nan, inplace=False)
        #df_clean = df_clean.dropna(subset=[f"{attr}.text.curated", f"{attr}.llm"])
        #df_clean = df_clean.replace(np.nan,"n/a", inplace=False).replace("None","n/a", inplace=False)
        df_clean = df_clean.applymap(
        lambda x: (
            "; ".join(sorted(set(i.strip() for i in str(x).split(";") if i.strip())))
            if pd.notnull(x) else x
        )
    )
        # Eval match between digit characters only
        df_clean = df_clean.applymap(
        lambda x: re.sub(r"\D", "", str(x)) if pd.notnull(x) else x
    )
        accuracy_llm = (df_clean[f"{attr}.text.curated"] == df_clean[f"{attr}.llm"]).sum()/len(df_clean)*100

        accuracy_annot = (df_clean[f"{attr}.text.0"] == df_clean[f"{attr}.text.1"]).sum()/len(df_clean)*100

        print(f'{attr} & {accuracy_annot:.2f} & n/a & {accuracy_llm:.2f} & n/a')
eval_all_span()

TimeOfOffense & 83.66 & n/a & 94.55 & n/a
DateOfOffense & 85.64 & n/a & 99.50 & n/a
MeasuringTime & 75.25 & n/a & 81.19 & n/a
AmountUsed & 76.73 & n/a & 90.10 & n/a


## Validation

In [184]:
allowed = {"SubstanceUsed":["alcohol", "cannabis", "methamphetamine", "amphetamine", "cocaine", "mdma", "hashish", "morphine", "other", "n/a"],
          "ReasonForCheck": ["traffic_accident", "speeding", "traffic_violation", "suspected_intoxication", "traffic_check", "other", "n/a"],
          "VehicleType": ["car", "van", "truck", "bus", "motorcycle", "other", "n/a"],
          "LicenseStatus": ["banned", "none", "valid", "n/a"],
          "Measuringmethods": ["blood", "breath", "urine", "n/a"],
          "RefusedTesting": ["yes", "no"]

}


In [217]:
for attr in ["ReasonForCheck", "VehicleType", "LicenseStatus", "SubstanceUsed", "Measuringmethods", "RefusedTesting"]:
    val = sum(df_combined[f"{attr}.llm"].apply(
        lambda x: any(tag.strip() in allowed[attr] for tag in x.split(";"))
    ))
    print(f"{val/len(df_combined)*100} % validated for attribute {attr}")

1.0 % validated for attribute ReasonForCheck
1.0 % validated for attribute VehicleType
1.0 % validated for attribute LicenseStatus
0.8960396039603961 % validated for attribute SubstanceUsed
1.0 % validated for attribute Measuringmethods
1.0 % validated for attribute RefusedTesting


## Discrepancies

In [444]:
cols = [f"{attr}.value.0", f"{attr}.value.1", f"{attr}.llm"]
subset = df[cols + ["text"]]  # keep those 3 + the text column

# Keep only rows where not all values are the same
mask = subset[cols].nunique(axis=1) > 1
discrepancies = subset[mask]

display(discrepancies)

,VehicleType.value.0,VehicleType.value.1,VehicleType.llm,text
0,car,n/a,car,"dňa 07.11.2022 v čase o 06:55 hod. v meste V. na ulici S.. V.. Š. po ceste II. triedy č. 532 viedol ako vodič osobné motorové vozidlo značky Škoda Octavia Combi, evidenčného čísla RA595AZ, po tom, čo požil alkoholické nápoje, pričom pred areálom spoločnosti MOSS.SK, s. r. o. Revúca bol zastavený a kontrolovaný hliadkou Okresného riaditeľstva Policajného zboru SR, Okresného dopravného inšpektorátu Revúca, kde vykonanou dychovou skúškou na zistenie prítomnosti alkoholu analyzátorom dychu typu „Dräger Alcotest 7510 classic“ výrobného čísla ARMF-0489 mu dňa 07.11.2022 o 06:56 hod. bolo nameraných 0,74 mg etanolu na liter vydychovaného vzduchu, čo je 1,54 promile,"
7,n/a,car,car,"dňa XX. XX. XXXX v čase o XX.XX hod. viedol ako vodič osobné motorové vozidlo zn. BMW ev. č. T. po ceste I. triedy č. XX po ul. Š. v meste C., kde bol predpísaným spôsobom zastavovaný, pričom na tieto výzvy vôbec nereagoval, naopak zrýchlil a v jazde pokračoval ďalej až na ul. R., kde pred rodinným domom č. 271/2 vozidlo zastavil, vyskočil z vozidla a začal utekať preč, pričom bol následne hliadkou OO PZ v Tornali spacifikovaný a pri služobnom zákroku bol použitý varovný výstrel do vzduchu, donucovacie prostriedky hmaty, chmaty a služobné putá a pri kontrole sa na výzvu policajta dňa XX. XX. XXXX v čase o XX.XX hod. odmietol podrobiť dychovej skúške na zistenie, či nie je ovplyvnený návykovou látkou analyzátorom dychu AlcoQuant 6020 plus výr. č. A409739 a taktiež sa odmietol podrobiť lekárskemu vyšetreniu odberom a vyšetrením krvi alebo iného biologického materiálu, hoci by to pri vyšetrení nebolo spojené s nebezpečenstvom pre jeho zdravie, následne pred umiestnením do cely policajného zaistenia zriadenej na OO PZ v Revúcej bol dňa XX. XX. XXXX v čase o XX.XX hod. podrobený dyhovej skúške analyzátorom dychu AlcoQuant 6020 plus výr. č. A408962, ktorým mu bola nameraná hodnota 0,71 miligramu etanolu na liter vydychovaného vzduchu, čo je pozitívny výsledok,"
9,n/a,car,car,", ž e - dňa 17.12.2021 v čase približne o 19:00 hod. viedol ako vodič motorové vozidlo zn. C. S., ev. č. S. po ceste II/519 v smere od obce Slovenské Pravno do obce Rudno, kde pri kilometrovníku 13,0 zišiel s vozidlom mimo cestu vľavo, pričom z miesta odišiel a následne, keď sa k motorovému vozidlu zn. C. S., ev. č. S. vrátil, bol po riadnom poučení vyzvaný, aby sa podrobil dychovej skúške na zistenie požitia alkoholu do elektronického prístroja Alcotest 7510, výrobné číslo ARMF-0427, kde metódou merania etanolu vo vydychovanom vzduchu mu bola dňa 17.12.2021 v čase o 20:47 hod. nameraná hodnota 1,07 mg/l alkoholu vo vydychovanom vzduchu, čo v prepočte predstavuje 2,23 ‰, opakovanou dychovou skúškou dňa 17.12.2021 v čase o 21:03 hod. nameraná hodnota 1,10 mg/1 alkoholu vo vydychovanom vzduchu, čo v prepočte predstavuje 2,29 ‰ a ďalšou opakovanou dychovou skúškou dňa 17.12.2021 v čase o 21:19 hod. nameraná hodnota 1,01 mg/l alkoholu vo vydychovanom vzduchu, čo v prepočte predstavuje 2,10 ‰, pričom na základe odborného vyjadrenia znalca prof. MUDr. Ľ. X., PhD. zo dňa 30.12.2021 vyplýva, že koncentrácia alkoholu v krvi obvineného bola dňa 17.12.2021 v čase o 19:00 hod. minimálne vo výške 1,33 až 1,48 ‰,"
12,n/a,car,car,", ž e dňa 06. novembra 2022 v čase okolo 22:40 hod. na miestnej komunikácii v obci Kluknava z parkoviska pod kaštieľom na parkovisko pred predajňou CBA a následne po parkovisku pred predajňou CBA, po predchádzajúcom požití alkoholu, ktorým sa priviedol do stavu vylučujúceho spôsobilosť bezpečne viesť motorové vozidlá, viedol osobné motorové vozidlo továrenskej značky Volkswagen Golf, evidenčného čísla J. XXX F., pričom dychovou skúškou elektronickým prístrojom AlcoQuant 6020 plus mu bola nameraná hodnota 1,27 mg/l alkoholu v dychu,"
13,n/a,car,car,"dňa XX.XX.XXXX v čase približne o XX:XX hod. v obci G. Y. smerom na obec S. riadil osobné motorové vozidlo zn. S. D., ev. č. BN XXX CH, kde bol pred domom č.

### Span Attributes

In [421]:
attr = "DateOfOffense"
df_clean = df.replace("n/a", np.nan, inplace=False)
df_clean = df_clean.dropna(subset=[f"{attr}.text.0",f"{attr}.text.1", f"{attr}.llm"])


In [422]:
print(f"Evaluating all {len(df)} records")
print((df[f"{attr}.text.0"] == df[f"{attr}.text.1"]).sum()/len(df))
print((df[f"{attr}.text.0"] == df[f"{attr}.llm"]).sum()/len(df))
print((df[f"{attr}.text.1"] == df[f"{attr}.llm"]).sum()/len(df))

Evaluating all 202 records
0.8267326732673267
0.8811881188118812
0.8762376237623762


In [423]:
print(f"Evaluating only non-na {len(df_clean)} records")
print((df_clean[f"{attr}.text.0"] == df_clean[f"{attr}.text.1"]).sum()/len(df_clean))
print((df_clean[f"{attr}.text.0"] == df_clean[f"{attr}.llm"]).sum()/len(df_clean))
print((df_clean[f"{attr}.text.1"] == df_clean[f"{attr}.llm"]).sum()/len(df_clean))

Evaluating only non-na 167 records
0.9880239520958084
0.9640718562874252
0.9580838323353293


### Discrepancies

In [487]:
cols = [f"{attr}.text.0", f"{attr}.text.1", f"{attr}.llm"]
subset = df[cols + ["text"]]  # keep those 3 + the text column

# Keep only rows where not all values are the same
mask = subset[cols].nunique(axis=1) > 1
discrepancies = subset[mask]

display(discrepancies)

,TypeOfRoad.text.0,TypeOfRoad.text.1,TypeOfRoad.llm,text
0,ceste II. triedy,ceste II. triedy č. 532,class_II,"dňa 07.11.2022 v čase o 06:55 hod. v meste V. na ulici S.. V.. Š. po ceste II. triedy č. 532 viedol ako vodič osobné motorové vozidlo značky Škoda Octavia Combi, evidenčného čísla RA595AZ, po tom, čo požil alkoholické nápoje, pričom pred areálom spoločnosti MOSS.SK, s. r. o. Revúca bol zastavený a kontrolovaný hliadkou Okresného riaditeľstva Policajného zboru SR, Okresného dopravného inšpektorátu Revúca, kde vykonanou dychovou skúškou na zistenie prítomnosti alkoholu analyzátorom dychu typu „Dräger Alcotest 7510 classic“ výrobného čísla ARMF-0489 mu dňa 07.11.2022 o 06:56 hod. bolo nameraných 0,74 mg etanolu na liter vydychovaného vzduchu, čo je 1,54 promile,"
1,n/a,n/a,urban,"v Košiciach na ul. G. v smere od ul. F. E., dňa 1.11.2022 v čase okolo 02:37 hod. v stave vylučujúcom spôsobilosť viedol osobné motorové vozidlo zn. L. A., EVČ: L. XXXG., kde bol zastavený a kontrolovaný hliadkou Mestskej polície Košice a následne hliadkou OOPZ Košice - Západ, pričom bol podrobený dychovej skúške na určenie objemového percenta alkoholu v dychu prístrojom AlcoQuat 6020 plus, výrobné číslo A408921, pri ktorej mu bola nameraná pozitívna hodnota 0.58 mg/l,"
2,n/a,ulici Pionierska,urban,"dňa 31.10.2022 v čase o 01:30 hod. po predchádzajúcom požití alkoholických nápojov, bez dokladov potrebných k vedeniu a prevádzke motorového vozidla viedla osobné motorové vozidlo zn. Volkswagen Passat, ev. č. E. XXX C. po ulici Pionierska v Rajeckých Tepliciach, v smere jazdy od ulice Školskej ovplyvnená alkoholom v čase o 01:34 hod. vo výške 1,35 mg etanolu na liter vydychovaného vzduchu (čo predstavuje 2,81 promile), opakovanou dychovou skúškou jej bola v čase o 02:04 nameraná hodnota vo výške 1,45 mg etanolu na liter vydychovaného vzduchu (čo predstavuje 3,02 promile) a v čase o 03:10 hod. vo výške 1,20 mg etanolu na litere vydychovaného vzduchu (čo predstavuje 2,50 promile), pričom svojím konaním porušila ust. §4 ods. 1 písm. b); § 4 ods. 2 písm. d) Zákona č. 8/2009 Z.z. o cestnej premávke,"
3,n/a,ulici Poľná,n/a,", ž e - dňa 26. októbra 2022 po predchádzajúcom požití alkoholických nápojov viedol ako vodič motocykel značky Defender, bez evidenčného čísla v obci Tomášovce po ulici Poľná v smere na Ulicu SNP, kde bol pred rodinným domom číslo XXX v čase o 09:55 hodine zastavený a kontrolovaný hliadkou OR PZ ODI v Lučenci a počas kontroly bol vyzvaný, aby sa podrobil dychovej skúške na zistenie prítomnosti alkoholu v jeho dychu, čo po riadnom poučení vykonal a elektronickým prístrojom AlcoQuant 6020plus, evidenčného čísla A409594 mu bola dňa 26. októbra 2022 v čase o 10,00 hodine nameraná hodnota 1,15 miligramu na liter vydychovaného etanolu v dychu,"
5,n/a,n/a,urban,"dňa XX.XX.XXXX v čase o XX:XX V.. po predchádzajúcom požití alkoholických nápojov viedol osobné motorové vozidlo značky K. L. ev. č.: O., po ulici H.. L. P. T., kde bol príslušníkmi OO PZ T.-P. pri bytovom dome č. XX, zastavený a okrem iného vyzvaný k podrobeniu sa dychovej skúške na zistenie prítomnosti alkoholu vo vydychovanom vzduch prístrojom H. XXXX, výrobné číslo H. XXXX, čo i vykonal a v dychu obvineného bola zistená prítomnosť alkoholu v čase o XX:XX hod. v objeme 0,86 mg/l (1,79 ‰) a v čase o 20:25 hod. v objeme 0,81 mg/l (1,69 ‰),"
6,n/a,ulici J.,urban,"dňa 24.10.2022 v čase o 14.20 h v meste Q. okr. Q. po predchádzajúcom požití alkoholických nápojov, viedla ako vodička osobné motorové vozidlo továrenskej značky M. N., Y.. Č..: Q., L.. G., v meste Q. po ulici J. C. v smere od U. E. smerom na Q. D., kde bola pri firme E. E. zákonným spôsobom zastavená a kontrolovaná hliadkou Oddelenia železničnej Polície Zvolen, pričom pri kontrole bola poučená a vyzvaná, aby sa podrobila dychovej skúške analyzátorom dychu AlcoQuant 6020 plus, evidenčné číslo A409058, ktorým sa určí objemové percento alkoholu v dychu, ktorú neodmietla a tejto sa dobrovoľne podrobila a dychovou skúškou jej bola